# Оценка baseline RAG и агента — шаг 9

Текущий эксперимент: **noreason-v2**. Сначала читаем сохранённые ответы и ручные оценки, затем сравниваем документальные ответы через RAGAS.

Используйте окружение **sklearn-agent**. Совместимость RAGAS зафиксирована в requirements-eval.txt: python -m pip install -r requirements-eval.txt.

**Run All безопасен по умолчанию:** новые HTTP-запросы и обращения к judge выключены. Включайте их явно через переключатели ниже. Старые результаты сохраняются; для новой модели или промпта задайте другой RUN_NAME.

In [1]:
"""Prepare paths and environment for agent evaluation."""

import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


current = Path.cwd().resolve()

PROJECT_ROOT = next(
    path
    for path in (current, *current.parents)
    if (path / "app" / "config.py").is_file()
    and (path / "notebooks" / "rag_eval.ipynb").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env", override=False)

DATASET_VERSION = "agent_eval_v1"

print(f"Python: {sys.executable}")
print(f"Project: {PROJECT_ROOT}")
print(f"Dataset: {DATASET_VERSION}")

COLLECT_NEW_RESPONSES = False
RUN_RAGAS = False


pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", "{:.3f}".format)


Python: C:\Users\belon\anaconda3\envs\sklearn-agent\python.exe
Project: C:\mentoring\sklearn-agent-service
Dataset: agent_eval_v1


## 1. Датасет

10 обычных вопросов (один из них о возможностях) и 5 multi-hop задач. Эталоны не передаются отвечающей модели. Постоянные ID связывают ответы, trace и оценки.

In [2]:
SINGLE_HOP = [
    {
        "id": "sh-01",
        "question": "How does Ridge regression handle multicollinearity?",
        "reference": (
            "Ridge adds an L2 penalty to the loss, shrinking coefficients "
            "and making estimation less sensitive to multicollinearity."
        ),
    },
    {
        "id": "sh-02",
        "question": "What does the alpha parameter control in Ridge?",
        "reference": (
            "Alpha controls regularization strength. A larger alpha "
            "increases the penalty on coefficient magnitude."
        ),
    },
    {
        "id": "sh-03",
        "question": "What is the difference between Lasso and Ridge?",
        "reference": (
            "Lasso uses an L1 penalty and can produce sparse coefficients. "
            "Ridge uses an L2 penalty and generally shrinks coefficients "
            "without producing sparsity."
        ),
    },
    {
        "id": "sh-04",
        "question": "What does min_samples_leaf control in a decision tree?",
        "reference": (
            "It sets the minimum number of samples required in each leaf. "
            "Larger values constrain splits and can reduce overfitting."
        ),
    },
    {
        "id": "sh-05",
        "question": "When does a decision tree overfit?",
        "reference": (
            "An overly complex tree can fit noise in the training data. "
            "Depth limits, minimum sample constraints and pruning "
            "can help control overfitting."
        ),
    },
    {
        "id": "sh-06",
        "question": "What is the formula for precision?",
        "reference": (
            "Precision = TP / (TP + FP). It is the fraction of positive "
            "predictions that are actually positive."
        ),
    },
    {
        "id": "sh-07",
        "question": "When is recall more important than precision?",
        "reference": (
            "Recall is especially important when false negatives "
            "are costly, because it measures the fraction of actual "
            "positives that are detected."
        ),
    },
    {
        "id": "sh-08",
        "question": "Что такое L2-регуляризация?",
        "reference": (
            "L2-регуляризация добавляет штраф, пропорциональный "
            "сумме квадратов коэффициентов модели. "
            "Ridge использует L2-регуляризацию."
        ),
    },
    {
        "id": "sh-09",
        "question": "Что такое F1-мера?",
        "reference": (
            "F1 — гармоническое среднее precision и recall: "
            "F1 = 2 * precision * recall / (precision + recall)."
        ),
    },
    {
        "id": "sh-10",
        "question": "Что ты умеешь?",
        "reference": (
            "Помощник предназначен для классического машинного обучения "
            "и scikit-learn. Индекс охватывает выбранные разделы: "
            "линейные модели, деревья и ансамбли, метрики и "
            "кросс-валидацию, предобработку, пайплайны, подбор параметров, "
            "обработку пропусков и отбор признаков. "
            "Это не вся документация scikit-learn."
        ),
    },
]

MULTI_HOP = [
    {
        "id": "mh-01",
        "question": (
            "Найди в документации формулу L2-штрафа Ridge. "
            "Затем через python_repl вычисли только этот штраф "
            "для alpha=0.5 и w=[1, 2, 3], без ошибки предсказаний."
        ),
        "reference": (
            "Штраф равен alpha * sum(w_i**2). "
            "Для заданных значений: 0.5 * (1 + 4 + 9) = 7."
        ),
        "expected_value": 7.0,
    },
    {
        "id": "mh-02",
        "question": (
            "Find the precision formula in the documentation. "
            "Then use python_repl to compute precision "
            "for TP=8 and FP=2."
        ),
        "reference": "Precision = TP / (TP + FP) = 8 / 10 = 0.8.",
        "expected_value": 0.8,
    },
    {
        "id": "mh-03",
        "question": (
            "Найди в документации формулу recall. Затем используй "
            "python_repl для расчёта recall при TP=8 и FN=4."
        ),
        "reference": "Recall = TP / (TP + FN) = 8 / 12 = 2/3.",
        "expected_value": 2 / 3,
    },
    {
        "id": "mh-04",
        "question": (
            "Find the F1 formula in the documentation. Then use "
            "python_repl to compute F1 for precision=0.8 and recall=0.6."
        ),
        "reference": (
            "F1 = 2 * precision * recall / (precision + recall). "
            "The result is 24/35, approximately 0.685714."
        ),
        "expected_value": 24 / 35,
    },
    {
        "id": "mh-05",
        "question": (
            "Find the mean squared error formula in the documentation. "
            "Then use python_repl to compute MSE for "
            "y_true=[1, 2, 3] and y_pred=[1, 2, 5]."
        ),
        "reference": (
            "MSE is the mean of squared prediction errors. "
            "Here MSE = (0 + 0 + 4) / 3 = 4/3."
        ),
        "expected_value": 4 / 3,
    },
]

ALL = [
    {
        **item,
        "group": "single_hop",
        "expected_tool_sequence": [],
    }
    for item in SINGLE_HOP
] + [
    {
        **item,
        "group": "multi_hop",
        "expected_tool_sequence": [
            "documentation_search",
            "python_repl",
        ],
    }
    for item in MULTI_HOP
]

In [3]:
"""Validate the dataset before making any API calls."""

assert len(SINGLE_HOP) == 10
assert len(MULTI_HOP) == 5
assert len(ALL) == 15

ids = [item["id"] for item in ALL]
assert len(ids) == len(set(ids)), "Question IDs must be unique"

for item in ALL:
    assert 1 <= len(item["question"]) <= 500, item["id"]
    assert item["question"].strip(), item["id"]
    assert item["reference"].strip(), item["id"]

    if item["group"] == "multi_hop":
        assert isinstance(item["expected_value"], (int, float))
        assert item["expected_tool_sequence"] == [
            "documentation_search",
            "python_repl",
        ]

dataset_preview = pd.DataFrame([
    {
        "id": item["id"],
        "group": item["group"],
        "question": item["question"],
        "expected_value": item.get("expected_value"),
    }
    for item in ALL
])

print(
    f"Готово: {len(SINGLE_HOP)} обычных вопросов "
    f"+ {len(MULTI_HOP)} multi-hop = {len(ALL)}"
)

display(dataset_preview)

Готово: 10 обычных вопросов + 5 multi-hop = 15


,id,group,question,expected_value
0,sh-01,single_hop,How does Ridge regression handle multicollinearity?,NaN
1,sh-02,single_hop,What does the alpha parameter control in Ridge?,NaN
2,sh-03,single_hop,What is the difference between Lasso and Ridge?,NaN
3,sh-04,single_hop,What does min_samples_leaf control in a decision tree?,NaN
4,sh-05,single_hop,When does a decision tree overfit?,NaN
5,sh-06,single_hop,What is the formula for precision?,NaN
6,sh-07,single_hop,When is recall more important than precision?,NaN
7,sh-08,single_hop,Что такое L2-регуляризация?,NaN
8,sh-09,single_hop,Что такое F1-мера?,NaN
9,sh-10,single_hop,Что ты умеешь?,NaN


## 2. Сохранённые ответы

Метка модели из окружения ноутбука не подтверждает модель запущенного сервера. Для нового сбора запустите API с нужными настройками и включите COLLECT_NEW_RESPONSES. Успешный ответ HTTP не означает правильное решение.

In [4]:
"""Prepare a resumable evaluation run."""

import json
import os
import time
import uuid
from datetime import datetime, timezone

import httpx

from app.schemas.agent import AgentResponse
from app.schemas.chat import ChatResponse


BASE_URL = "http://127.0.0.1:8000"
INTERVAL_S = 20

RUN_NAME = "noreason-v2"

RUN_DIR = PROJECT_ROOT / "notebooks" / "agent_eval_runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = RUN_DIR / "responses.json"

if RESULTS_PATH.exists():
    run = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))

    if run["questions"] != ALL:
        raise ValueError(
            "The dataset changed. Choose a new RUN_NAME."
        )
else:
    if not COLLECT_NEW_RESPONSES:
        raise FileNotFoundError("Saved responses missing; enable collection for a new run.")
    run = {
        "run_name": RUN_NAME,
        "dataset_version": DATASET_VERSION,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "local_model_label": os.getenv("LLM_MODEL", "not specified"),
        "questions": ALL,
        "results": {},
    }


def save_run() -> None:
    RESULTS_PATH.write_text(
        json.dumps(run, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


print(f"Results: {RESULTS_PATH}")
print(f"Previously saved requests: {len(run['results'])}")
print(f"Local model label: {run['local_model_label']}")

Results: C:\mentoring\sklearn-agent-service\notebooks\agent_eval_runs\noreason-v2\responses.json
Previously saved requests: 30
Local model label: qwen3.6-35b-a3b-noreason


In [5]:
"""Collect baseline and agent responses, saving each completed request."""

SYSTEMS = {
    "baseline": ("/chat", ChatResponse),
    "agent": ("/agent", AgentResponse),
}


def collect_responses(questions: list[dict]) -> None:
    with httpx.Client(
        base_url=BASE_URL,
        timeout=180.0,
    ) as client:
        ready = client.get("/ready")
        ready.raise_for_status()

        for item in questions:
            for system, (endpoint, response_schema) in SYSTEMS.items():
                key = f"{system}:{item['id']}"
                previous = run["results"].get(key)

                if previous and previous["status"] == "ok":
                    print(f"Skip {key}: already saved")
                    continue

                payload = {"question": item["question"]}

                if system == "agent":
                    payload["thread_id"] = (
                        f"eval-{RUN_NAME}-{item['id']}-{uuid.uuid4().hex}"
                    )

                started = time.perf_counter()
                response = None

                try:
                    response = client.post(endpoint, json=payload)
                    response.raise_for_status()

                    data = response_schema.model_validate(
                        response.json()
                    ).model_dump()

                except (httpx.HTTPError, ValueError) as exc:
                    run["results"][key] = {
                        "system": system,
                        "question_id": item["id"],
                        "status": "error",
                        "error_type": type(exc).__name__,
                        "http_status": (
                            response.status_code
                            if response is not None
                            else None
                        ),
                        "client_elapsed_ms": int(
                            (time.perf_counter() - started) * 1000
                        ),
                    }
                    save_run()

                    print(
                        f"Stopped at {key}: {type(exc).__name__}. "
                        "Completed responses have been saved."
                    )
                    return

                run["results"][key] = {
                    "system": system,
                    "question_id": item["id"],
                    "status": "ok",
                    "completed_at": datetime.now(timezone.utc).isoformat(),
                    "client_elapsed_ms": int(
                        (time.perf_counter() - started) * 1000
                    ),
                    "response": data,
                }
                save_run()

                print(f"Saved {key}")
                time.sleep(INTERVAL_S)

In [6]:
if COLLECT_NEW_RESPONSES:
    collect_responses(ALL)

received = sum(record["status"] == "ok" for record in run["results"].values())
print(f"Saved API responses: {received} / {2 * len(ALL)}")

Saved API responses: 30 / 30


## 3. Поведение, источники и время

Порядок названий инструментов ещё не доказывает успешное решение. Trace обрезает результат инструмента до 500 символов; для проверки используем оригинальные full_context. Время клиента не включает паузы между запросами.

In [7]:
"""Inspect saved responses without calling either system."""

run = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))


def contains_sequence(actual: list[str], expected: list[str]) -> bool:
    """Check whether expected tools appear in the required order."""
    position = 0

    for tool in actual:
        if position < len(expected) and tool == expected[position]:
            position += 1

    return position == len(expected)


rows = []

for item in run["questions"]:
    for system in SYSTEMS:
        key = f"{system}:{item['id']}"
        record = run["results"].get(key, {})
        received = record.get("status") == "ok"
        data = record.get("response", {}) if received else {}

        tools = [
            step["tool"]
            for step in data.get("trace", [])
            if step.get("tool")
        ]

        is_agent_multi_hop = (
            system == "agent"
            and item["group"] == "multi_hop"
        )

        rows.append({
            "key": key,
            "question_id": item["id"],
            "group": item["group"],
            "system": system,
            "response_received": received,
            "client_elapsed_ms": record.get("client_elapsed_ms"),
            "iterations": data.get("iterations"),
            "tools": tools,
            "expected_order_present": (
                received
                and contains_sequence(
                    tools,
                    item["expected_tool_sequence"],
                )
                if is_agent_multi_hop
                else None
            ),
            "guardrail": data.get("guardrail_triggered"),
            "termination": data.get("termination_reason"),
        })

results_df = pd.DataFrame(rows)

display(results_df.drop(columns="tools").set_index("key"))

,question_id,group,system,response_received,client_elapsed_ms,iterations,expected_order_present,guardrail,termination
key,,,,,,,,,
baseline:sh-01,sh-01,single_hop,baseline,True,2740,NaN,None,NaN,None
agent:sh-01,sh-01,single_hop,agent,True,8345,2.000,None,output_too_long,None
baseline:sh-02,sh-02,single_hop,baseline,True,3812,NaN,None,NaN,None
agent:sh-02,sh-02,single_hop,agent,True,9943,2.000,None,NaN,None
baseline:sh-03,sh-03,single_hop,baseline,True,4326,NaN,None,NaN,None
agent:sh-03,sh-03,single_hop,agent,True,10014,2.000,None,output_too_long,None
baseline:sh-04,sh-04,single_hop,baseline,True,3554,NaN,None,NaN,None
agent:sh-04,sh-04,single_hop,agent,True,18302,2.000,None,NaN,None
baseline:sh-05,sh-05,single_hop,baseline,True,3344,NaN,None,NaN,None


In [8]:
"""Display one question with references, answers and agent trace."""

questions_by_id = {
    item["id"]: item
    for item in run["questions"]
}


def inspect_question(question_id: str, *, show_sources: bool = False) -> None:
    item = questions_by_id[question_id]

    print(f"Question: {item['question']}")
    print(f"\nReference: {item['reference']}")

    for system in SYSTEMS:
        key = f"{system}:{question_id}"
        record = run["results"].get(key)

        print(f"\n{'=' * 60}")
        print(system.upper())

        if not record or record["status"] != "ok":
            print("No valid API response")
            continue

        data = record["response"]
        print(data["answer"])

        print(f"\nGuardrail: {data.get('guardrail_triggered')}")
        print(f"Termination: {data.get('termination_reason')}")

        if system == "agent":
            display(pd.DataFrame(data["trace"]))

        if not show_sources:
            continue
        for index, source in enumerate(data["sources"], start=1):
            print(f"\nSource {index}: {source['url']}")
            print(source["full_context"])

## 4. Ручная оценка multi-hop

Оценки относятся только к выбранному RUN_NAME и хранятся в manual_reviews.json.

- numerical_correct: верно итоговое число; разумное округление допустимо.
- answer_correct: математический/ML ответ достаточно полон и не содержит существенных ложных объяснений. Найденная формула в trace удовлетворяет просьбе «найди формулу»; отсутствие формулы и цитат в итоговом тексте отмечено отдельно.
- tool_chain_valid: исходный документ поддерживает формулу, поиск выполнен перед расчётом, Python использует заданные значения и возвращает правильный результат.
- Успешность задачи также требует ожидаемого порядка инструментов, отсутствия ограничения итераций и замены ответа guardrail.

Заявления baseline о выполнении Python и язык ответа отмечаются отдельно в notes, а не включаются в математическую точность. Непроверенные оценки не считаются нулями и не переносятся между экспериментами. Все пояснения сохранены в JSON; таблица ниже показывает краткие итоги.


In [9]:
"""Load persistent manual reviews for multi-hop questions."""

REVIEWS_PATH = RUN_DIR / "manual_reviews.json"
if REVIEWS_PATH.exists():
    manual_reviews = json.loads(REVIEWS_PATH.read_text(encoding="utf-8"))
else:
    manual_reviews = {
        f"{system}:{item['id']}": {
            "answer_correct": None,
            "numerical_correct": None,
            "tool_chain_valid": None,
            "notes": "",
        }
        for item in run["questions"] if item["group"] == "multi_hop"
        for system in SYSTEMS
    }


def save_reviews() -> None:
    REVIEWS_PATH.write_text(
        json.dumps(manual_reviews, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )


save_reviews()
display(pd.DataFrame.from_dict(manual_reviews, orient="index").drop(columns="notes"))


,answer_correct,numerical_correct,tool_chain_valid
baseline:mh-01,True,True,None
agent:mh-01,True,True,True
baseline:mh-02,True,True,None
agent:mh-02,True,True,True
baseline:mh-03,True,True,None
agent:mh-03,True,True,True
baseline:mh-04,True,True,None
agent:mh-04,False,True,True
baseline:mh-05,True,True,None
agent:mh-05,True,True,True


In [10]:
"""Summarize reviewed multi-hop results without provider calls."""

multi_hop_ids = [
    item["id"] for item in run["questions"]
    if item["group"] == "multi_hop"
]
for question_id in multi_hop_ids:
    for system in SYSTEMS:
        key = f"{system}:{question_id}"
        assert run["results"].get(key, {}).get("status") == "ok", key
        assert type(manual_reviews[key]["answer_correct"]) is bool, key
        assert type(manual_reviews[key]["numerical_correct"]) is bool, key
        assert manual_reviews[key]["notes"].strip(), key
        if system == "agent":
            assert type(manual_reviews[key]["tool_chain_valid"]) is bool, key

n_multi_hop = len(multi_hop_ids)
assert n_multi_hop > 0


def accuracy(system: str, field: str) -> float:
    return sum(
        manual_reviews[f"{system}:{question_id}"][field]
        for question_id in multi_hop_ids
    ) / n_multi_hop


agent_tasks_passed = 0
for question_id in multi_hop_ids:
    key = f"agent:{question_id}"
    review = manual_reviews[key]
    data = run["results"][key]["response"]
    tools = [step["tool"] for step in data["trace"] if step.get("tool")]
    expected = questions_by_id[question_id]["expected_tool_sequence"]
    agent_tasks_passed += bool(
        review["answer_correct"] and review["tool_chain_valid"]
        and contains_sequence(tools, expected)
        and not data.get("termination_reason")
        and not data.get("guardrail_triggered")
    )

multi_hop_summary = {
    "n_questions": n_multi_hop,
    "baseline_numerical_accuracy": accuracy("baseline", "numerical_correct"),
    "agent_numerical_accuracy": accuracy("agent", "numerical_correct"),
    "baseline_answer_accuracy": accuracy("baseline", "answer_correct"),
    "agent_answer_accuracy": accuracy("agent", "answer_correct"),
    "agent_tool_chain_valid_rate": accuracy("agent", "tool_chain_valid"),
    "agent_task_success_rate": agent_tasks_passed / n_multi_hop,
}
(RUN_DIR / "multi_hop_metrics.json").write_text(
    json.dumps(multi_hop_summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
display(pd.DataFrame.from_dict(multi_hop_summary, orient="index", columns=["value"]))


,value
n_questions,5.000
baseline_numerical_accuracy,1.000
agent_numerical_accuracy,1.000
baseline_answer_accuracy,1.000
agent_answer_accuracy,0.800
agent_tool_chain_valid_rate,1.000
agent_task_success_rate,0.800


## 5. RAGAS: девять документационных вопросов

Faithfulness проверяет поддержку утверждений исходным контекстом; AnswerRelevancy — соответствие вопросу. Это не проверка отдельных цитат, языка ответа, всех фактов или исполнения инструментов.

Мета-вопрос о возможностях исключён из документального grounding. Multi-hop оценивается отдельно: документы поддерживают формулу, Python — вычисление. Обрезанные ответы оцениваются такими, какими их получил пользователь.

До 54 обращений к judge при strictness=1, с паузой 20 секунд. Готовые отчёты читаются с диска. Одинаковая модель генерации и оценки может давать общие смещения.

При временных ошибках разрешены до двух повторов только сбойного вопроса. Первый успешный балл сохраняется; не выбираем лучший из повторов. Ошибки квоты и неопределённые метрики не повторяются автоматически. Исходные и повторные отчёты сохранены отдельно; source_report показывает происхождение каждого балла.


In [11]:
"""Prepare all nine documentation questions for paired RAGAS evaluation."""

import hashlib
from concurrent.futures import ThreadPoolExecutor

from langchain_openai import ChatOpenAI

from app.config import settings
from app.rag.embeddings import get_embeddings
from app.scripts.eval_ragas import run_evaluation


run = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))

DOCUMENTATION_IDS = [
    item["id"]
    for item in run["questions"]
    if item["group"] == "single_hop"
    and item["id"] != "sh-10"
]

assert len(DOCUMENTATION_IDS) == 9

ragas_rows = {
    "baseline": {},
    "agent": {},
}

for item in run["questions"]:
    if item["id"] not in DOCUMENTATION_IDS:
        continue

    for system in SYSTEMS:
        key = f"{system}:{item['id']}"
        record = run["results"][key]

        assert record["status"] == "ok", key

        data = record["response"]
        assert data["answer"].strip(), key
        assert data["sources"], key

        sources = []

        for source in data["sources"]:
            assert source["full_context"].strip(), key

            sources.append({
                "url": source["url"],
                "content": source["full_context"],
            })

        ragas_rows[system][item["id"]] = {
            "id": item["id"],
            "question": item["question"],
            "answer": data["answer"],
            "sources": sources,
            "expected_refusal": False,
        }

print(f"Prepared: {len(DOCUMENTATION_IDS)} questions per system")
from importlib.metadata import version
from urllib.parse import urlsplit

JUDGE_MODEL = settings.llm_model
INPUT_SHA256 = hashlib.sha256(RESULTS_PATH.read_bytes()).hexdigest()
DRIVER_SHA256 = hashlib.sha256((PROJECT_ROOT / "app/scripts/eval_ragas.py").read_bytes()).hexdigest()
EVALUATION_VERSIONS = {name: version(name) for name in (
    "ragas", "langchain-community", "langchain-experimental", "langchain-core",
    "langchain-openai", "sentence-transformers",
)}
BATCHES = [DOCUMENTATION_IDS[index:index + 3] for index in range(0, 9, 3)]
judge = None
judge_embeddings = None


Prepared: 9 questions per system


In [12]:
"""Load saved judge reports or explicitly evaluate a batch."""
def _evaluate_batch(system: str, question_ids: list[str], batch_name: str) -> dict | None:
    global judge, judge_embeddings
    assert system in ragas_rows and 1 <= len(question_ids) <= 6
    assert len(question_ids) == len(set(question_ids))
    assert all(question_id in DOCUMENTATION_IDS for question_id in question_ids)
    output = RUN_DIR / f"ragas_{system}_{batch_name}.json"
    parameters = {
        "run_name": RUN_NAME, "system": system, "question_ids": question_ids,
        "provider_host": urlsplit(settings.llm_base_url).hostname,
        "judge_model": JUDGE_MODEL, "temperature": 0, "max_output_tokens": 4096,
        "embedding_model": settings.embedding_model,
        "normalize_embeddings": settings.normalize_embeddings,
        "strictness": 1, "interval_s": 20, "sdk_retries": 0, "ragas_retries": 0,
        "input_sha256": INPUT_SHA256, "driver_sha256": DRIVER_SHA256,
        "versions": EVALUATION_VERSIONS,
        "limitations": [
            "Same model may generate and evaluate answers; shared bias is possible.",
            "Faithfulness does not validate individual citations or factual correctness.",
            "Nine documentation questions only; multi-hop is reviewed separately.",
            "Truncated answers are evaluated as delivered.",
        ],
    }
    if output.exists():
        report = json.loads(output.read_text(encoding="utf-8"))
        if report["parameters"] != parameters:
            raise ValueError(f"Settings changed for {output.name}; use a new batch name.")
        return report
    if not RUN_RAGAS:
        print(f"Not evaluated: {output.name} (RUN_RAGAS=False)")
        return None
    if judge is None:
        judge = ChatOpenAI(
            base_url=settings.llm_base_url, api_key=settings.llm_api_key,
            model=JUDGE_MODEL, temperature=0, max_retries=0, timeout=120,
            max_tokens=4096, streaming=False,
        ).bind(response_format={"type": "json_object"})
        judge_embeddings = get_embeddings(device="cpu")
    rows = [ragas_rows[system][question_id] for question_id in question_ids]
    with ThreadPoolExecutor(max_workers=1) as executor:
        report = executor.submit(
            run_evaluation, rows, judge, judge_embeddings, output,
            parameters=parameters, call_budget=3 * len(rows), interval_s=20,
        ).result()
    print(f"{output.name}: complete={report['complete']}, calls={report['calls_attempted']}")
    return report

def _recover_batch(system: str, batch_name: str, report: dict) -> dict:
    """Retry transient failures only; preserve the first completed score."""
    effective = json.loads(json.dumps(report))
    parent = f"ragas_{system}_{batch_name}.json"
    for question in effective["questions"]:
        for metric in question["metrics"].values():
            metric.setdefault("source_report", parent)
    if effective["complete"] or effective.get("stop_reason") == "provider_limit":
        return effective
    effective["recovery_reports"] = []
    retryable = {
        "OpenAIAPIError", "OpenAIAPIConnectionError", "OpenAIAPITimeoutError",
        "APITimeoutError", "APIConnectionError", "TimeoutError", "ReadTimeout",
        "ConnectError", "RagasOutputParserException",
    }
    for attempt in (1, 2):
        for question in effective["questions"]:
            failed = [
                metric for metric in question["metrics"].values()
                if metric["status"] == "error" and metric.get("error_type") in retryable
            ]
            if not failed:
                continue
            retry_name = f"{batch_name}-retry-{attempt}-{question['id']}"
            retry = _evaluate_batch(system, [question["id"]], retry_name)
            if retry is None:
                continue
            filename = f"ragas_{system}_{retry_name}.json"
            effective["recovery_reports"].append(filename)
            for candidate in retry["questions"]:
                assert candidate["id"] == question["id"]
                for name, metric in candidate["metrics"].items():
                    if question["metrics"][name]["status"] != "completed" and metric["status"] == "completed":
                        question["metrics"][name] = {**metric, "source_report": filename}
            if retry.get("stop_reason") == "provider_limit":
                effective["stop_reason"] = "provider_limit"
                break
        if effective.get("stop_reason") == "provider_limit":
            break
    effective["complete"] = all(
        metric["status"] == "completed"
        for question in effective["questions"] for metric in question["metrics"].values()
    )
    if effective["complete"]:
        effective["stop_reason"] = None
    if effective["recovery_reports"]:
        (RUN_DIR / f"ragas_{system}_{batch_name}-effective.json").write_text(
            json.dumps(effective, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
        )
    return effective


def evaluate_batch(system: str, question_ids: list[str], batch_name: str) -> dict | None:
    report = _evaluate_batch(system, question_ids, batch_name)
    return _recover_batch(system, batch_name, report) if report is not None else None

reports = {system: [] for system in SYSTEMS}
for index, question_ids in enumerate(BATCHES, start=1):
    for system in SYSTEMS:
        report = evaluate_batch(system, question_ids, f"batch-{index}")
        if report is not None:
            reports[system].append(report)
            if RUN_RAGAS and not report["complete"]:
                raise RuntimeError(f"Incomplete batch: {system}, {index}; inspect the saved report.")

## 6. Итоговое сравнение

RAGAS сравнивается по одинаковым вопросам: показываем число пар и статусы каждой метрики. Отсутствующий балл не равен нулю. Ручные multi-hop оценки относятся к другим пяти вопросам и не смешиваются со средним RAGAS.

Время и число обрезаний рассчитаны по всему набору. Записанные результаты не означают, что агент гарантированно лучше baseline.

В этом прогоне Faithfulness агента ниже: judge часто не находит в полученных документах поддержку дополнительных примеров и объяснений. Для sh-07 это прикладные примеры стоимости ошибок, для sh-02 — детали поведения Ridge, отсутствующие в выбранных фрагментах. Это не доказывает, что все такие утверждения ложны: даже правильный самостоятельный пример может получить низкий document grounding. Извлечение утверждений и вердикты judge тоже могут ошибаться.

После первого HTTP 502 сохранён отдельный повторный отчёт. Для каждой метрики используем первый успешный балл; путь к его отчёту находится в ragas_scores.json. Первичный неудачный отчёт не удалён, а восстановленный пакет доступен как batch-1-effective.json.


In [13]:
score_rows = []
for system, system_reports in reports.items():
    for report in system_reports:
        for item in report["questions"]:
            row = {"system": system, "question_id": item["id"]}
            for metric in ("faithfulness", "answer_relevancy"):
                result = item["metrics"][metric]
                row[metric] = result["value"]
                row[f"{metric}_status"] = result["status"]
                row[f"{metric}_source_report"] = result.get("source_report")
            score_rows.append(row)
ragas_df = pd.DataFrame(score_rows, columns=[
    "system", "question_id", "faithfulness", "faithfulness_status",
    "answer_relevancy", "answer_relevancy_status",
    "faithfulness_source_report", "answer_relevancy_source_report",
])
assert not ragas_df.duplicated(["system", "question_id"]).any()
if all((ragas_df[f"{metric}_status"] == "completed").all()
       for metric in ("faithfulness", "answer_relevancy")) and len(ragas_df):
    paired_scores = ragas_df.pivot(
        index="question_id", columns="system",
        values=["faithfulness", "answer_relevancy"],
    )
    display(paired_scores.style.format("{:.3f}").bar(
        vmin=0, vmax=1, color="#d8eaf5", align="left",
    ))
else:
    display(ragas_df.set_index(["question_id", "system"]))

ragas_summary = {}
comparison_rows = []
for metric in ("faithfulness", "answer_relevancy"):
    completed = ragas_df[ragas_df[f"{metric}_status"] == "completed"]
    paired = completed.pivot(index="question_id", columns="system", values=metric)
    if all(system in paired.columns for system in SYSTEMS):
        paired = paired.dropna(subset=list(SYSTEMS))
    else:
        paired = pd.DataFrame(columns=list(SYSTEMS))
    ragas_summary[metric] = {
        "n_paired": len(paired), "n_expected": len(DOCUMENTATION_IDS),
        **{system: float(paired[system].mean()) if len(paired) else None for system in SYSTEMS},
    }
    comparison_rows.append({
        "Metric": f"{metric} (documentation)",
        "Baseline RAG": ragas_summary[metric]["baseline"],
        "Agent": ragas_summary[metric]["agent"], "N paired": len(paired),
    })
ragas_complete = all(value["n_paired"] == 9 for value in ragas_summary.values())
for metric, baseline_key, agent_key in (
    ("Numerical accuracy (multi-hop)", "baseline_numerical_accuracy", "agent_numerical_accuracy"),
    ("Math/ML answer accuracy (multi-hop)", "baseline_answer_accuracy", "agent_answer_accuracy"),
):
    comparison_rows.append({
        "Metric": metric, "Baseline RAG": multi_hop_summary[baseline_key],
        "Agent": multi_hop_summary[agent_key], "N paired": multi_hop_summary["n_questions"],
    })
comparison_rows.append({
    "Metric": "Task success (multi-hop)", "Baseline RAG": None,
    "Agent": multi_hop_summary["agent_task_success_rate"], "N paired": 5,
})
comparison_rows.append({
    "Metric": "Mean HTTP time, ms (all 15)",
    **{"Baseline RAG" if system == "baseline" else "Agent":
       float(results_df.loc[results_df.system == system, "client_elapsed_ms"].mean())
       for system in SYSTEMS}, "N paired": 15,
})
comparison_rows.append({
    "Metric": "Mean agent iterations (all 15)", "Baseline RAG": None,
    "Agent": float(results_df.loc[results_df.system == "agent", "iterations"].mean()), "N paired": 15,
})
comparison_rows.append({
    "Metric": "Truncated answers (all 15)", "Baseline RAG": None,
    "Agent": int((results_df.guardrail == "output_too_long").sum()), "N paired": 15,
})
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.style.format(precision=3, na_rep="—"))
print(f"RAGAS complete: {ragas_complete}")

RAGAS complete: True


,Metric,Baseline RAG,Agent,N paired
0,faithfulness (documentation),0.926,0.470,9
1,answer_relevancy (documentation),0.940,0.942,9
2,Numerical accuracy (multi-hop),1.000,1.000,5
3,Math/ML answer accuracy (multi-hop),1.000,0.800,5
4,Task success (multi-hop),—,0.800,5
5,"Mean HTTP time, ms (all 15)",3981.600,8546.600,15
6,Mean agent iterations (all 15),—,2.267,15
7,Truncated answers (all 15),—,5.000,15


In [14]:
"""Export scores and a comparison suitable for the README."""
summary_payload = {
    "run_name": RUN_NAME, "ragas_complete": ragas_complete,
    "ragas": ragas_summary, "multi_hop": multi_hop_summary,
    "input_sha256": INPUT_SHA256, "judge_model": JUDGE_MODEL,
    "evaluation_versions": EVALUATION_VERSIONS,
}
for filename, payload in (
    ("ragas_scores.json", json.loads(ragas_df.to_json(orient="records"))),
    ("comparison_metrics.json", summary_payload),
):
    (RUN_DIR / filename).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
    )

def markdown_table(frame: pd.DataFrame) -> str:
    def value(item):
        return "—" if pd.isna(item) else f"{item:.3f}" if isinstance(item, float) else str(item)
    lines = ["| " + " | ".join(frame.columns) + " |", "| " + " | ".join(["---"] * len(frame.columns)) + " |"]
    lines += ["| " + " | ".join(value(item) for item in row) + " |" for row in frame.itertuples(index=False, name=None)]
    return "\n".join(lines)

notes = (
    "\n\n## Interpretation limits\n\n"
    "- RAGAS covers nine documentation questions; multi-hop is reviewed separately on five cases.\n"
    "- Each answer is judged against its original context; retrieved contexts may differ between systems.\n"
    "- Same-model judging, strictness=1 and one run limit confidence.\n"
    "- Faithfulness does not validate individual citations, tool execution, language or all facts.\n"
    "- Baseline execution-like wording is excluded from mathematical accuracy; see manual_reviews.json.\n"
    "- Five agent single-hop answers were truncated; delivered answers remain in the evaluation.\n"
    "- The agent F1 number is correct, but the claim that it is lower than both 0.8 and 0.6 is false.\n"
    "- The capabilities question is not included in documentation grounding.\n"
    "- One HTTP 502 was recovered with a question-level retry; original failed and retry reports remain saved.\n"
    "- Only the first completed score per metric is used; source_report identifies its origin.\n"
    "\n## Observed results\n\n"
    "Baseline Faithfulness is about 0.926, agent about 0.470; relevancy is about 0.940 vs 0.942.\n"
    "The agent adds more examples and general explanations not established by its retrieved chunks.\n"
    "For sh-07 the judge marked application-specific cost examples unsupported; for sh-02 the retrieved alpha context lacked several claimed Ridge details.\n"
    "Low document support does not establish factual error: valid illustrative calculations and general knowledge may be penalized too.\n"
    "This judge's statement extraction and verdicts can be imperfect; inspect the raw reports before treating a score as an objective truth.\n"
)
(RUN_DIR / "comparison.md").write_text(
    f"# Agent vs baseline RAG — {RUN_NAME}\n\nRAGAS complete: {ragas_complete}\n\n"
    + markdown_table(comparison_df) + notes, encoding="utf-8",
)
print(f"Reports saved in {RUN_DIR.relative_to(PROJECT_ROOT)}")

Reports saved in notebooks\agent_eval_runs\noreason-v2


## Повторный запуск

Оставьте сетевые переключатели выключенными для просмотра готовой оценки. Для новой модели или промпта создайте другой RUN_NAME, соберите обе системы заново и выполните новые ручные оценки.

При незавершённом отчёте сначала посмотрите status, error_type и stop_reason. Не удаляйте его и не подменяйте пропущенные оценки нулём. Оценка качества запускается отдельно от CI-тестов.